# 16 — MLP trained with **intersection only** (sum of min)

**Advisor direction (Dr. Prasad / Buddhi):** Use the **min / intersection** term
\(I(A,B) = \sum_i \min(A_i, B_i)\). Avoid the **ratio** \(I / (\|A\|+\|B\|-I)\) in training
because min/max gradients can be troublesome.

**This notebook (clean build):**
- Train **from scratch** — no `best_model` / cosine checkpoint
- Loss: triplet on **intersection only** on 512-D embeddings (ReLU + L1)
- Stage-1 search: top-K by **intersection** (GPU), not WJ-ratio HNSW
- Stage-2 (optional): exact **raw** WJ ratio rerank for GT comparison only

**Prereqs:** `/tmp/qt_10k.npy`, `/tmp/gt_lookup_10k.pkl` from `00_cache_data.ipynb`

In [1]:
# ── Configuration ───────────────────────────────────────────────────────────
dataset_name = "10k"       # "10k" only for now (full needs log1p + more RAM)
run_training = True
run_eval = True
device_str = "cuda:0"

QUERY_START_10K = 8000
QUERY_START_FULL = 187019

max_pos = 30
batch_size = 512
epochs = 50
lr = 1e-3
weight_decay = 1e-4
intersection_margin = 0.01   # on sum(min) scale (vectors sum to 1)

candidate_ks = [500, 1000]
rerank_batch_size = 16
search_corpus_chunk = 2000   # GPU intersection matmul chunk size

ckpt_path = "/tmp/best_compressor_intersection_min_10k.pt"
out_path = "/tmp/results_mlp_intersection_min.pkl"
seed = 42

In [2]:
import gc
import os
import pickle
import random
import time
from pathlib import Path

import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

device = torch.device(device_str if torch.cuda.is_available() else "cpu")
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print(f"device={device} | dataset={dataset_name} | train from scratch (no warm-start)")

device=cuda:0 | dataset=10k | train from scratch (no warm-start)


In [3]:
# ── Model + intersection (min-only) loss ─────────────────────────────────────
class QuadtreeCompressorMin(nn.Module):
    """MLP → nonnegative L1-simplex embedding (mass / probability per dimension)."""

    def __init__(self, in_dim, out_dim=512, use_log1p=False):
        super().__init__()
        self.use_log1p = use_log1p
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )

    def forward(self, x):
        if self.use_log1p:
            x = torch.log1p(x * 1e6)
        out = self.net(x)
        out = F.relu(out)
        out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out


def intersection(a, b):
    """Buddhi I(A,B) = sum_i min(A_i, B_i). No ratio."""
    return torch.minimum(a, b).sum(dim=-1)


def intersection_triplet_loss(anchors, positives, margin=0.01):
    """
    Want I(anchor, positive) > I(anchor, hard_negative) + margin.
    Hard negative = highest intersection with another anchor's positive in batch.
    """
    i_pos = intersection(anchors, positives)

    # (B, B) cross intersections
    cross = torch.min(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(dim=2)
    cross.fill_diagonal_(-1e9)
    i_neg = cross.max(dim=1).values

    loss = F.relu(i_neg - i_pos + margin)
    violated = loss > 0
    if violated.sum() == 0:
        z = torch.tensor(0.0, device=anchors.device, requires_grad=True)
        return z, 0, i_pos.detach().mean(), i_neg.detach().mean()
    return (
        loss[violated].mean(),
        int(violated.sum().item()),
        i_pos.detach().mean(),
        i_neg.detach().mean(),
    )


class AnchorPositiveDataset(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, query_start, max_pos=30):
        self.vecs = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"Anchor-positive pairs: {len(self.pairs):,}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        qid, pos_id = self.pairs[idx]
        return self.vecs[qid], self.vecs[pos_id]


def raw_intersection_np(a, b):
    return float(np.minimum(a, b).sum())


def raw_wj_ratio_np(a, b):
    """Reference only — ratio used in GT / rerank, not in training."""
    mins = np.minimum(a, b).sum()
    maxs = np.maximum(a, b).sum()
    return float(mins / max(maxs, 1e-10))


print("Intersection-min model + loss defined.")

Intersection-min model + loss defined.


In [4]:
def load_dataset(name):
    if name == "10k":
        qt = np.load("/tmp/qt_10k.npy")
        with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
            gt = pickle.load(f)
        query_start = QUERY_START_10K
        use_log1p = False
    elif name == "full":
        qt = np.load("/tmp/qtree_vectors_full.npy")
        with open("/tmp/gt_lookup_full.pkl", "rb") as f:
            gt = pickle.load(f)
        query_start = QUERY_START_FULL
        use_log1p = True
        ckpt_path_full = "/tmp/best_compressor_intersection_min_full.pt"
    else:
        raise ValueError(name)

    corpus_qt = qt[:query_start]
    query_qt = qt[query_start:]
    corpus_sums = corpus_qt.sum(axis=1)
    print(f"qt={qt.shape} | corpus={corpus_qt.shape} | queries={query_qt.shape} | log1p={use_log1p}")
    return qt, gt, query_start, use_log1p, corpus_qt, query_qt, corpus_sums


qt, gt, query_start, use_log1p, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)

qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499) | log1p=False


In [5]:
def train_intersection_min(qt, gt, query_start, use_log1p, n_epochs):
    print("Training from scratch (random init). No cosine checkpoint.")
    model = QuadtreeCompressorMin(qt.shape[1], out_dim=512, use_log1p=use_log1p).to(device)

    dataset = AnchorPositiveDataset(qt, gt, query_start=query_start, max_pos=max_pos)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=device.type == "cuda",
        drop_last=True,
    )
    print(f"Steps/epoch: {len(loader)}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_loss = float("inf")
    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss = 0.0
        steps = 0
        pbar = tqdm(loader, desc=f"Epoch {epoch:02d}/{n_epochs}", leave=False)
        for anchor, positive in pbar:
            anchor = anchor.to(device, non_blocking=True)
            positive = positive.to(device, non_blocking=True)
            B = anchor.shape[0]
            combined = torch.cat([anchor, positive], dim=0)
            out = model(combined)
            a_emb = out[:B]
            p_emb = out[B:]

            loss, n_viol, i_pos, i_neg = intersection_triplet_loss(
                a_emb, p_emb, margin=intersection_margin
            )
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += float(loss.detach().cpu())
            steps += 1
            pbar.set_postfix(
                loss=f"{float(loss):.4f}",
                viol=n_viol,
                Ipos=f"{float(i_pos):.3f}",
                Ineg=f"{float(i_neg):.3f}",
            )

        avg_loss = total_loss / max(steps, 1)
        scheduler.step()
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), ckpt_path)

        if epoch == 1 or epoch % 5 == 0 or epoch == n_epochs:
            print(
                f"Epoch {epoch:02d}/{n_epochs} | loss={avg_loss:.4f} | best={best_loss:.4f} | "
                f"lr={scheduler.get_last_lr()[0]:.2e}"
            )

    print(f"Saved {ckpt_path} (best_loss={best_loss:.4f})")
    return model


if run_training:
    model = train_intersection_min(qt, gt, query_start, use_log1p, epochs)
else:
    model = QuadtreeCompressorMin(qt.shape[1], out_dim=512, use_log1p=use_log1p).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    print(f"Loaded {ckpt_path}")
model.eval()

Training from scratch (random init). No cosine checkpoint.
Anchor-positive pairs: 46,722
Steps/epoch: 91


Epoch 01/50 | loss=0.0164 | best=0.0164 | lr=9.99e-04


Epoch 05/50 | loss=0.0130 | best=0.0124 | lr=9.76e-04


Epoch 10/50 | loss=0.0117 | best=0.0114 | lr=9.05e-04


Epoch 15/50 | loss=0.0119 | best=0.0112 | lr=7.94e-04


...

Epoch 35/50 | loss=0.0106 | best=0.0106 | lr=2.06e-04


Epoch 40/50 | loss=0.0106 | best=0.0105 | lr=9.55e-05


Epoch 45/50 | loss=0.0105 | best=0.0105 | lr=2.45e-05


Epoch 50/50 | loss=0.0106 | best=0.0104 | lr=0.00e+00
Saved /tmp/best_compressor_intersection_min_10k.pt (best_loss=0.0104)


QuadtreeCompressorMin(
  (net): Sequential(
    (0): Linear(in_features=18499, out_features=4096, bias=False)
    (1): BatchNorm1d(4096, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=4096, out_features=1024, bias=False)
    (4): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=512, bias=False)
    (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)

In [10]:
def generate_embeddings(model, data, *, dev, batch_size=512):
    """Pass dev as keyword: generate_embeddings(model, qt, dev=device)."""
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(data), batch_size), desc="Embedding"):
            batch = torch.tensor(data[start:start + batch_size], dtype=torch.float32, device=dev)
            chunks.append(model(batch).cpu().numpy())
    embs = np.vstack(chunks)
    print(f"simplex sums: min={embs.sum(1).min():.4f} max={embs.sum(1).max():.4f}")
    return embs


def embedding_quality_check(model, qt, gt, query_start, n=200):
    qids = [q for q in gt if q >= query_start][:n]
    raw_i_gt, raw_i_rd, raw_j_gt, emb_i_gt, emb_i_rd = [], [], [], [], []
    for qid in qids:
        pos_list = [p for p in gt[qid] if p < query_start]
        if not pos_list:
            continue
        pos_id = pos_list[0]
        rand_id = random.randrange(0, query_start)
        qv, pv, rv = qt[qid], qt[pos_id], qt[rand_id]
        raw_i_gt.append(raw_intersection_np(qv, pv))
        raw_i_rd.append(raw_intersection_np(qv, rv))
        raw_j_gt.append(raw_wj_ratio_np(qv, pv))
        with torch.no_grad():
            eq = model(torch.tensor(qv, dtype=torch.float32, device=device).unsqueeze(0))
            ep = model(torch.tensor(pv, dtype=torch.float32, device=device).unsqueeze(0))
            er = model(torch.tensor(rv, dtype=torch.float32, device=device).unsqueeze(0))
        emb_i_gt.append(float(intersection(eq, ep).cpu()))
        emb_i_rd.append(float(intersection(eq, er).cpu()))
    ri_gt, ri_rd = np.mean(raw_i_gt), np.mean(raw_i_rd)
    print(
        f"Raw  I(sum min) — GT: {ri_gt:.6f} | Rand: {ri_rd:.6f} | Gap: {ri_gt - ri_rd:.6f}"
        f"  (tiny on 18k cells; use J ratio for raw scale)"
    )
    print(
        f"Emb  I(sum min) — GT: {np.mean(emb_i_gt):.4f} | Rand: {np.mean(emb_i_rd):.4f} | "
        f"Gap: {np.mean(emb_i_gt) - np.mean(emb_i_rd):.4f}  (train metric)"
    )
    print(
        f"Raw  J ratio     — GT: {np.mean(raw_j_gt):.4f}  (GT / rerank only; not trained)"
    )


def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total = 0.0
    count = 0
    for i, ids in enumerate(nbrs):
        qid = query_start_id + i
        gt_set = set(gt_lookup.get(qid, [])[:k])
        if not gt_set:
            continue
        total += len(gt_set & set(ids[:k])) / len(gt_set)
        count += 1
    return total / count if count else 0.0


def eval_recall_dict(gt_lookup, nbrs, query_start_id, max_k):
    return {
        k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
        for k in (10, 50, 100, 500)
        if k <= max_k
    }


@torch.no_grad()
def knn_intersection_gpu(query_embs, corpus_embs, k, dev, corpus_chunk=2000):
    """Top-k by I(q,c) = sum(min(q,c)) — same metric as training."""
    q = torch.from_numpy(query_embs).to(dev, dtype=torch.float32)
    c_all = torch.from_numpy(corpus_embs).to(dev, dtype=torch.float32)
    n_q, n_c = q.shape[0], c_all.shape[0]
    top_ids = np.zeros((n_q, k), dtype=np.int64)
    top_scores = np.full((n_q, k), -1.0, dtype=np.float32)

    for qs in tqdm(range(0, n_q, 64), desc="KNN intersection"):
        qe = min(qs + 64, n_q)
        qb = q[qs:qe]
        best_scores = torch.full((qb.shape[0], k), -1.0, device=dev)
        best_ids = torch.zeros((qb.shape[0], k), dtype=torch.long, device=dev)

        for cs in range(0, n_c, corpus_chunk):
            ce = min(cs + corpus_chunk, n_c)
            cb = c_all[cs:ce]
            # (Bq, Bc, D) -> (Bq, Bc)
            scores = torch.min(qb[:, None, :], cb[None, :, :]).sum(dim=2)
            cand_ids = torch.arange(cs, ce, device=dev).expand(qb.shape[0], -1)
            merged_scores = torch.cat([best_scores, scores], dim=1)
            merged_ids = torch.cat([best_ids, cand_ids], dim=1)
            new_scores, order = torch.topk(merged_scores, k=min(k, merged_scores.shape[1]), dim=1)
            best_ids = torch.gather(merged_ids, 1, order)
            best_scores = new_scores

        top_ids[qs:qe] = best_ids.cpu().numpy()
        top_scores[qs:qe] = best_scores.cpu().numpy()

    nbrs = [(top_ids[i].tolist(), top_scores[i].tolist()) for i in range(n_q)]
    return nbrs


def rerank_wj_gpu(query_qt, nbrs_ids, corpus_qt, corpus_sums, dev, batch_size=16):
    """Exact raw quadtree WJ ratio — for GT-aligned evaluation only."""
    corpus_t = torch.from_numpy(corpus_qt).to(device=dev, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=dev, dtype=torch.float32)
    reranked = []
    for start in tqdm(range(0, len(nbrs_ids), batch_size), desc="Raw WJ ratio rerank"):
        batch = nbrs_ids[start:start + batch_size]
        groups = {}
        for offset, ids in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for cand_len, items in groups.items():
            if cand_len == 0:
                for _, _ in items:
                    reranked.append([])
                continue
            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[abs_i] for abs_i, _ in items], axis=0)
            ids_t = torch.from_numpy(ids_np).to(device=dev)
            q_t = torch.from_numpy(query_np).to(device=dev, dtype=torch.float32)
            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order = torch.argsort(mins / maxs.clamp_min(1e-10), dim=1, descending=True).cpu().numpy()
            for row, (_, ids) in zip(order, items):
                reranked.append(ids[row].tolist())
    return reranked

In [11]:
if run_eval:
    if Path(ckpt_path).exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()

    print("\n" + "=" * 72)
    print("MLP INTERSECTION-MIN — eval (train & search = sum(min) only)")
    print("=" * 72)

    embedding_quality_check(model, qt, gt, query_start)

    embs = generate_embeddings(model, qt, dev=device)
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]

    results = {}
    max_k = max(max(candidate_ks), 500)

    t0 = time.time()
    nbrs_min = knn_intersection_gpu(
        query_embs, corpus_embs, k=max_k, dev=device, corpus_chunk=search_corpus_chunk
    )
    qps_min = len(query_embs) / (time.time() - t0)
    ids_only = [ids for ids, _ in nbrs_min]

    print(f"\n--- Stage 1: top-{max_k} by INTERSECTION (sum min) on 512-D ---")
    rec = eval_recall_dict(gt, ids_only, query_start, max_k)
    results["intersection_min_no_rerank"] = {**rec, "qps": qps_min}
    for k, r in rec.items():
        print(f"  R@{k:<4} = {r:.4f}")
    print(f"  QPS ≈ {qps_min:.1f}")

    for k in candidate_ks:
        print(f"\n--- Stage 2: top-{k} intersection candidates + raw WJ RATIO rerank ---")
        cand_ids = [ids[:k] for ids, _ in nbrs_min]
        t0 = time.time()
        rr_ids = rerank_wj_gpu(query_qt, cand_ids, corpus_qt, corpus_sums, device, rerank_batch_size)
        qps = len(query_embs) / (time.time() - t0)
        rec_rr = eval_recall_dict(gt, rr_ids, query_start, k)
        results[f"k{k}_raw_wj_ratio_rerank"] = {**rec_rr, "qps": qps, "k": k}
        for rk, rv in rec_rr.items():
            print(f"  R@{rk:<4} = {rv:.4f}")
        print(f"  QPS ≈ {qps:.1f}")

    payload = {dataset_name: results, "_meta": {
        "train": "intersection_triplet_sum_min",
        "search_stage1": "intersection_sum_min",
        "rerank": "raw_wj_ratio_for_gt_only",
        "init": "scratch",
        "ckpt": ckpt_path,
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }}
    with open(out_path, "wb") as f:
        pickle.dump(payload, f)
    print(f"\nSaved {out_path}")

    print("\n--- Compare to notebook 15 (WJ ratio train + WJ HNSW) ---")
    print("  15 random-init:  R@10 ~0.56 no rerank, ~0.99 K1000+rerank")
    print("  02 cosine MLP:   R@10 ~0.67 no rerank, ~0.997 K500+rerank")

gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()


MLP INTERSECTION-MIN — eval (train & search = sum(min) only)
Raw  I(sum min) — GT: 0.000006 | Rand: 0.000001 | Gap: 0.000006  (tiny on 18k cells; use J ratio for raw scale)
Emb  I(sum min) — GT: 0.9934 | Rand: 0.9492 | Gap: 0.0443  (train metric)
Raw  J ratio     — GT: 0.7996  (GT / rerank only; not trained)


Embedding: 100%|██████████| 20/20 [00:00<00:00, 219.71it/s]


simplex sums: min=1.0000 max=1.0000


KNN intersection: 100%|██████████| 32/32 [00:00<00:00, 201.12it/s]



--- Stage 1: top-1000 by INTERSECTION (sum min) on 512-D ---
  R@10   = 0.6518
  R@50   = 0.7729
  R@100  = 0.8052
  R@500  = 0.9062
  QPS ≈ 7614.2

--- Stage 2: top-500 intersection candidates + raw WJ RATIO rerank ---


Raw WJ ratio rerank: 100%|██████████| 125/125 [00:00<00:00, 161.65it/s]


  R@10   = 0.9916
  R@50   = 0.9917
  R@100  = 0.9872
  R@500  = 0.9062
  QPS ≈ 2501.1

--- Stage 2: top-1000 intersection candidates + raw WJ RATIO rerank ---


Raw WJ ratio rerank: 100%|██████████| 125/125 [00:01<00:00, 82.35it/s]


  R@10   = 0.9920
  R@50   = 0.9930
  R@100  = 0.9925
  R@500  = 0.9688
  QPS ≈ 1291.7

Saved /tmp/results_mlp_intersection_min.pkl

--- Compare to notebook 15 (WJ ratio train + WJ HNSW) ---
  15 random-init:  R@10 ~0.56 no rerank, ~0.99 K1000+rerank
  02 cosine MLP:   R@10 ~0.67 no rerank, ~0.997 K500+rerank
